<a href="https://colab.research.google.com/github/sudhirpol522/CuteDSL/blob/main/Cutedsl_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!python --version
!which ncu || true
!which nsys || true
!which compute-sanitizer || true

Tue Aug 25 00:20:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Install CuTe DSL

Run this before importing `cutlass`. The current default wheel targets CUDA Toolkit 12.9 and requires an NVIDIA driver compatible with it. If compilation later reports an insufficient driver or unsupported PTX version, the Colab host driver is the blocker; it cannot be upgraded from inside the notebook.

In [ ]:
%pip install -q --upgrade nvidia-cutlass-dsl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 29.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch

N = 1000
THREADS = 128

@cute.kernel
def copy_kernel(
    src: cute.Tensor,
    dst: cute.Tensor,
    n: cutlass.Int32,
):
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    block_size, _, _ = cute.arch.block_dim()

    idx = bx * block_size + tx

    if idx < n:
        dst[idx] = src[idx]


@cute.jit
def copy_tensor(
    src: cute.Tensor,
    dst: cute.Tensor,
    n: cutlass.Int32,
):
    copy_kernel(src, dst, n).launch(
        block=(THREADS, 1, 1),
        grid=(cute.ceil_div(n, THREADS), 1, 1),
    )


cutlass.cuda.initialize_cuda_context()

x = torch.ones(N, dtype=torch.float32, device="cuda")
y = torch.empty_like(x)


x_cute = from_dlpack(x,assumed_align=16) # he bytes ahe 4 bytes , 8 bytes, 16 bytes etc
y_cute = from_dlpack(y,assumed_align=16)

copy_tensor(x_cute, y_cute, N)

torch.testing.assert_close(y, x)
print(y)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 

In [ ]:
x

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import cutlass.cute.runtime as cute_rt
import torch


@cute.jit
def copy_from_ptr(
    ptr: cute.Pointer,
    dst: cute.Tensor,
):
    layout = cute.make_layout((8, 5), stride=(5, 1))
    src = cute.make_tensor(ptr, layout)

    cute.basic_copy(src, dst)


a = torch.arange(40, dtype=torch.float32).reshape(8, 5)
b = torch.empty_like(a)

ptr_a = cute_rt.make_ptr(
    cutlass.Float32,
    a.data_ptr(),
)

copy_from_ptr(ptr_a, from_dlpack(b))

print("a:")
print(a)

print("b:")
print(b)

torch.testing.assert_close(b, a)

a:
tensor([[ 0.,  1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.,  9.],
        [10., 11., 12., 13., 14.],
        [15., 16., 17., 18., 19.],
        [20., 21., 22., 23., 24.],
        [25., 26., 27., 28., 29.],
        [30., 31., 32., 33., 34.],
        [35., 36., 37., 38., 39.]])
b:
tensor([[ 0.,  1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.,  9.],
        [10., 11., 12., 13., 14.],
        [15., 16., 17., 18., 19.],
        [20., 21., 22., 23., 24.],
        [25., 26., 27., 28., 29.],
        [30., 31., 32., 33., 34.],
        [35., 36., 37., 38., 39.]])


In [ ]:
import ctypes
import torch
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack


@cute.jit
def tensor_access_item(a: cute.Tensor):
    identity = cute.make_identity_tensor(a.layout.shape)

    cute.printf(
        "a[2] = {} (equivalent to a[{}])",
        a[2],
        identity[2],
    )
    cute.printf(
        "a[9] = {} (equivalent to a[{}])",
        a[9],
        identity[9],
    )

    cute.printf("a[2,0] = {}", a[2, 0])
    cute.printf("a[2,4] = {}", a[2, 4])

    a[2, 3] = 100.0
    a[2, 4] = 101.0

    cute.printf("a[2,3] = {}", a[2, 3])
    cute.printf("a[(2,4)] = {}", a[(2, 4)])

    print(f"Compile-time layout: {a.layout}")


data = torch.arange(40, dtype=torch.float32).reshape(8, 5)

tensor_access_item(from_dlpack(data))

libc = ctypes.CDLL(None)
torch.cuda.synchronize()
libc.fflush(None)

print(data)

Compile-time layout: (8,5):(5,1)
tensor([[  0.,   1.,   2.,   3.,   4.],
        [  5.,   6.,   7.,   8.,   9.],
        [ 10.,  11.,  12., 100., 101.],
        [ 15.,  16.,  17.,  18.,  19.],
        [ 20.,  21.,  22.,  23.,  24.],
        [ 25.,  26.,  27.,  28.,  29.],
        [ 30.,  31.,  32.,  33.,  34.],
        [ 35.,  36.,  37.,  38.,  39.]])


In [ ]:
@cute.kernel
def print_tensor_kernel(A: cute.Tensor):
    tidx, _, _ = cute.arch.thread_idx()
    cute.printf("hello")

    if tidx == 0:
        cute.printf("here")
        cute.print_tensor(A)


@cute.jit
def hello_world(A: cute.Tensor):
    zipped_22 = cute.zipped_divide(A, (2, 2))
    tiled_22 = cute.tiled_divide(A, (2, 2))
    zipped_2 = cute.zipped_divide(A, (2,))


    # Safe because (6,8) is divisible by (2,2)
    print_tensor_kernel(tiled_22).launch(
        grid=(1, 1, 1),
        block=(32, 1, 1),
    )


A = torch.arange(
    6 * 8,
    device="cuda",
    dtype=torch.float16,
).reshape(6, 8)

hello_world(from_dlpack(A, assumed_align=16))
torch.cuda.synchronize()

import ctypes
ctypes.CDLL(None).fflush(None)

0

In [ ]:
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

import torch


@cute.kernel
def local_partition_kernel(A: cute.Tensor, thread_layout: cute.Layout):
    tidx, _, _ = cute.arch.thread_idx()

    print(f"A is {A}") # (8,8):(8,1)

    print(f"thread layout: {thread_layout}") # (2, 4):(4, 1)

    thrA = cute.local_partition(A, thread_layout, tidx) # (4, 2):(16, 4)
    print(f"local partition for thrA is {thrA} for thread")
    cute.printf("thread {}: thrA = {}", tidx, thrA)

@cute.jit
def explain_local_partition(A: cute.Tensor):
    thread_layout = cute.make_layout((2, 4), stride=(4, 1))

    local_partition_kernel(A, thread_layout).launch(
        grid=(1, 1, 1),
        block=(cute.size(thread_layout), 1, 1),
    )


A = torch.arange(0, 8 * 8, device="cuda", dtype=torch.float32).reshape(8, 8)
tensor = from_dlpack(A, assumed_align=16)

explain_local_partition(tensor)
torch.cuda.synchronize()


A is tensor<ptr<f32, gmem, align<16>> o (8,8):(8,1)>
thread layout: (2,4):(4,1)
local partition for thrA is tensor<ptr<f32, gmem> o (4,2):(16,4)> for thread


In [ ]:
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def local_tile_2d_kernel(A: cute.Tensor):
    tiler = (4, 4)
    tile_coord = (1, 2)

    tile = cute.local_tile(A, tiler, tile_coord)

    # Compile-time information
    print("A:", A)
    print("2-D tiler:", tiler)
    print("tile coordinate:", tile_coord)
    print("selected tile:", tile)

    # Runtime GPU output
    cute.printf("2-D local_tile result:")
    cute.print_tensor(tile)


@cute.jit
def run_2d(A: cute.Tensor):
    local_tile_2d_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


A = torch.arange(
    16 * 16,
    device="cuda",
    dtype=torch.float32,
).reshape(16, 16)

run_2d(from_dlpack(A, assumed_align=16))
torch.cuda.synchronize()

A: tensor<ptr<f32, gmem, align<16>> o (16,16):(16,1)>
2-D tiler: (4, 4)
tile coordinate: (1, 2)
selected tile: tensor<ptr<f32, gmem, align<16>> o (4,4):(16,1)>


In [ ]:
@cute.kernel
def compare_local_tile_kernel(A: cute.Tensor):
    # Ordinary 2-D tiling
    tile_2d = cute.local_tile(
        A,
        tiler=(4, 4),
        coord=(1, 2),
    )

    # GEMM-style 3-D tiler projected onto A(M,K)
    tile_3d = cute.local_tile(
        A,
        tiler=(4, 8, 4),
        coord=(1, 0, 2),
        proj=(1, None, 1),
    )

    print("2-D result:", tile_2d)
    print("projected 3-D result:", tile_3d)

    cute.printf("Using the 2-D tiler:")
    cute.print_tensor(tile_2d)

    cute.printf("Using the projected GEMM tiler:")
    cute.print_tensor(tile_3d)


@cute.jit
def compare_local_tile(A: cute.Tensor):
    compare_local_tile_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )

nvidia-cutlass-dsl: 4.7.0
PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
GPU: NVIDIA A100-SXM4-40GB

===== EXAMPLE 1: BASIC (4,2) COMPOSITION =====
Example 1 A: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
Example 1 transform: (4,2):(2,1)
Example 1 composed: tensor<ptr<f32, gmem, align<16>> o (4,2):(2,1)>

===== EXAMPLE 2: REORDERED (4,2) COMPOSITION =====
Example 2 A: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
Example 2 transform: (4,2):(1,4)
Example 2 composed: tensor<ptr<f32, gmem, align<16>> o (4,2):(1,4)>

===== EXAMPLE 3: ALL 16 ELEMENTS AS (4,4) =====
Example 3 A: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
Example 3 transform: (4,4):(4,1)
Example 3 composed: tensor<ptr<f32, gmem, align<16>> o (4,4):(4,1)>

Host-side A (unchanged):
tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')


##Composition veg vgelya way ni use kru shakto. so veg vegle ways ahet document kelele, so te ekda bgh


In [ ]:
%%writefile /content/01_composition_reshape.py

import ctypes

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def reshape_kernel(A: cute.Tensor):
    # New 2-D coordinate system.
    #
    # transform(row, column) = row * 4 + column
    transform = cute.make_layout(
        shape=(4, 2),
        stride=(4, 2),
    )

    # C(row, column) = A(transform(row, column))
    C = cute.composition(A, transform)

    # These execute during kernel compilation.
    print("A descriptor:", A)
    print("transform layout:", transform)
    print("C descriptor:", C)

    # These execute on the GPU.
    cute.printf("Original 1-D tensor A:")
    cute.print_tensor(A)

    cute.printf("Composed 4x4 tensor C:")
    cute.print_tensor(C)

    cute.printf(
        "C[2,3] = {}, corresponding A[11] = {}",
        C[2, 3],
        A[11],
    )


@cute.jit
def run_reshape(A: cute.Tensor):
    reshape_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a CUDA GPU")

    A_torch = torch.arange(
        16,
        device="cuda",
        dtype=torch.float32,
    )

    A_cute = from_dlpack(
        A_torch,
        assumed_align=16,
    )

    run_reshape(A_cute)

    # Wait for the kernel and flush device printing.
    torch.cuda.synchronize()
    ctypes.CDLL(None).fflush(None)

    print("PyTorch tensor after the kernel:", A_torch, flush=True)


if __name__ == "__main__":
    main()

Overwriting /content/01_composition_reshape.py


In [ ]:
!python -u /content/01_composition_reshape.py

A descriptor: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
transform layout: (4,2):(4,2)
C descriptor: tensor<ptr<f32, gmem, align<16>> o (4,2):(4,2)>
Original 1-D tensor A:
tensor(raw_ptr(0x00007aaaa9200000: f32, gmem, align<16>) o (16):(1), data=
       [ 0.000000, ],
       [ 1.000000, ],
       [ 2.000000, ],
       ...
       [ 13.000000, ],
       [ 14.000000, ],
       [ 15.000000, ])
Composed 4x4 tensor C:
tensor(raw_ptr(0x00007aaaa9200000: f32, gmem, align<16>) o (4,2):(4,2), data=
       [[ 0.000000,  2.000000, ],
        [ 4.000000,  6.000000, ],
        [ 8.000000,  10.000000, ],
        [ 12.000000,  14.000000, ]])
C[2,3] = 14.000000, corresponding A[11] = 11.000000
PyTorch tensor after the kernel: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')


In [ ]:
%%writefile /content/02_composition_reorder.py

import ctypes

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def reorder_kernel(A: cute.Tensor):
    # Column-major-like coordinate mapping:
    #
    # transform(row, column) = row + 4 * column
    transform = cute.make_layout(
        shape=(4, 4),
        stride=(1, 4),
    )

    # C(row, column) = A(row + 4 * column)
    C = cute.composition(A, transform)

    # Compile-time descriptor information
    print("A descriptor:", A)
    print("transform layout:", transform)
    print("C descriptor:", C)

    # Runtime GPU output
    cute.printf("Original 1-D tensor A:")
    cute.print_tensor(A)

    cute.printf("Reordered 4x4 tensor C:")
    cute.print_tensor(C)

    cute.printf(
        "C[2,3] = {}, corresponding A[14] = {}",
        C[2, 3],
        A[14],
    )


@cute.jit
def run_reorder(A: cute.Tensor):
    reorder_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a CUDA GPU")

    A_torch = torch.arange(
        16,
        device="cuda",
        dtype=torch.float32,
    )

    A_cute = from_dlpack(
        A_torch,
        assumed_align=16,
    )

    run_reorder(A_cute)

    torch.cuda.synchronize()
    ctypes.CDLL(None).fflush(None)

    print("PyTorch tensor remains unchanged:", A_torch, flush=True)


if __name__ == "__main__":
    main()

Writing /content/02_composition_reorder.py


In [ ]:
!python -u /content/02_composition_reorder.py

A descriptor: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
transform layout: (4,4):(1,4)
C descriptor: tensor<ptr<f32, gmem, align<16>> o (4,4):(1,4)>
Original 1-D tensor A:
tensor(raw_ptr(0x000079d161200000: f32, gmem, align<16>) o (16):(1), data=
       [ 0.000000, ],
       [ 1.000000, ],
       [ 2.000000, ],
       ...
       [ 13.000000, ],
       [ 14.000000, ],
       [ 15.000000, ])
Reordered 4x4 tensor C:
tensor(raw_ptr(0x000079d161200000: f32, gmem, align<16>) o (4,4):(1,4), data=
       [[ 0.000000,  4.000000,  8.000000,  12.000000, ],
        [ 1.000000,  5.000000,  9.000000,  13.000000, ],
        [ 2.000000,  6.000000,  10.000000,  14.000000, ],
        [ 3.000000,  7.000000,  11.000000,  15.000000, ]])
C[2,3] = 14.000000, corresponding A[14] = 14.000000
PyTorch tensor remains unchanged: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')


In [ ]:
%%writefile /content/03_composition_select.py

import ctypes

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def select_kernel(A: cute.Tensor):
    # Select:
    #   2 elements from A's row mode
    #   3 elements from A's column mode
    #
    # A Shape can be used as a by-mode composition tiler.
    selector = (2, 3)

    selected = cute.composition(A, selector)

    # Compile-time descriptor information
    print("A descriptor:", A)
    print("selector:", selector)
    print("selected descriptor:", selected)

    # Runtime GPU values
    cute.printf("Original 4x4 tensor A:")
    cute.print_tensor(A)

    cute.printf("Selected upper-left 2x3 region:")
    cute.print_tensor(selected)

    cute.printf(
        "selected[1,2] = {}, corresponding A[1,2] = {}",
        selected[1, 2],
        A[1, 2],
    )


@cute.jit
def run_select(A: cute.Tensor):
    select_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a CUDA GPU")

    A_torch = torch.arange(
        16,
        device="cuda",
        dtype=torch.float32,
    ).reshape(4, 4)

    A_cute = from_dlpack(
        A_torch,
        assumed_align=16,
    )

    run_select(A_cute)

    torch.cuda.synchronize()
    ctypes.CDLL(None).fflush(None)

    print("PyTorch A remains unchanged:", flush=True)
    print(A_torch, flush=True)


if __name__ == "__main__":
    main()

Writing /content/03_composition_select.py


In [ ]:
!python -u /content/03_composition_select.py

A descriptor: tensor<ptr<f32, gmem, align<16>> o (4,4):(4,1)>
selector: (2, 3)
selected descriptor: tensor<ptr<f32, gmem, align<16>> o (2,3):(4,1)>
Original 4x4 tensor A:
tensor(raw_ptr(0x00007ca92d200000: f32, gmem, align<16>) o (4,4):(4,1), data=
       [[ 0.000000,  1.000000,  2.000000,  3.000000, ],
        [ 4.000000,  5.000000,  6.000000,  7.000000, ],
        [ 8.000000,  9.000000,  10.000000,  11.000000, ],
        [ 12.000000,  13.000000,  14.000000,  15.000000, ]])
Selected upper-left 2x3 region:
tensor(raw_ptr(0x00007ca92d200000: f32, gmem, align<16>) o (2,3):(4,1), data=
       [[ 0.000000,  1.000000,  2.000000, ],
        [ 4.000000,  5.000000,  6.000000, ]])
selected[1,2] = 6.000000, corresponding A[1,2] = 6.000000
PyTorch A remains unchanged:
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]], device='cuda:0')


In [ ]:
%%writefile /content/04_composition_stride.py

import ctypes

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def stride_kernel(A: cute.Tensor):
    # Eight output coordinates, separated by a stride of two.
    #
    # transform(i) = 2 * i
    transform = cute.make_layout(
        shape=(8,),
        stride=(2,),
    )

    # S(i) = A(transform(i)) = A(2*i)
    S = cute.composition(A, transform)

    # Compile-time descriptor information
    print("A descriptor:", A)
    print("transform layout:", transform)
    print("S descriptor:", S)

    # Runtime GPU values
    cute.printf("Original tensor A:")
    cute.print_tensor(A)

    cute.printf("Every second element:")
    cute.print_tensor(S)

    cute.printf(
        "S[5] = {}, corresponding A[10] = {}",
        S[5],
        A[10],
    )


@cute.jit
def run_stride(A: cute.Tensor):
    stride_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a CUDA GPU")

    A_torch = torch.arange(
        16,
        device="cuda",
        dtype=torch.float32,
    )

    A_cute = from_dlpack(
        A_torch,
        assumed_align=16,
    )

    run_stride(A_cute)

    torch.cuda.synchronize()
    ctypes.CDLL(None).fflush(None)

    print("PyTorch A remains unchanged:", A_torch, flush=True)


if __name__ == "__main__":
    main()

Writing /content/04_composition_stride.py


In [ ]:
!python -u /content/04_composition_stride.py

A descriptor: tensor<ptr<f32, gmem, align<16>> o (16):(1)>
transform layout: (8):(2)
S descriptor: tensor<ptr<f32, gmem, align<16>> o (8):(2)>
Original tensor A:
tensor(raw_ptr(0x00007d0681200000: f32, gmem, align<16>) o (16):(1), data=
       [ 0.000000, ],
       [ 1.000000, ],
       [ 2.000000, ],
       ...
       [ 13.000000, ],
       [ 14.000000, ],
       [ 15.000000, ])
Every second element:
tensor(raw_ptr(0x00007d0681200000: f32, gmem, align<16>) o (8):(2), data=
       [ 0.000000, ],
       [ 2.000000, ],
       [ 4.000000, ],
       ...
       [ 10.000000, ],
       [ 12.000000, ],
       [ 14.000000, ])
S[5] = 10.000000, corresponding A[10] = 10.000000
PyTorch A remains unchanged: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')


In [ ]:
%%writefile /content/05_composition_broadcast.py

import ctypes

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def broadcast_kernel(A: cute.Tensor):
    # B has shape (4, 3).
    #
    # transform(row, column) = row * 1 + column * 0
    #                        = row
    #
    # Therefore, changing the column does not change the
    # element selected from A.
    transform = cute.make_layout(
        shape=(4, 3),
        stride=(1, 0),
    )

    B = cute.composition(A, transform)

    # Compile-time descriptor information
    print("A descriptor:", A)
    print("transform layout:", transform)
    print("B descriptor:", B)

    # Runtime GPU values
    cute.printf("Original tensor A:")
    cute.print_tensor(A)

    cute.printf("Broadcast tensor B:")
    cute.print_tensor(B)

    cute.printf(
        "B[2,0]={}, B[2,1]={}, B[2,2]={}, A[2]={}",
        B[2, 0],
        B[2, 1],
        B[2, 2],
        A[2],
    )


@cute.jit
def run_broadcast(A: cute.Tensor):
    broadcast_kernel(A).launch(
        grid=(1, 1, 1),
        block=(1, 1, 1),
    )


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a CUDA GPU")

    A_torch = torch.arange(
        4,
        device="cuda",
        dtype=torch.float32,
    )

    A_cute = from_dlpack(
        A_torch,
        assumed_align=16,
    )

    run_broadcast(A_cute)

    torch.cuda.synchronize()
    ctypes.CDLL(None).fflush(None)

    print("PyTorch A remains unchanged:", A_torch, flush=True)


if __name__ == "__main__":
    main()

Writing /content/05_composition_broadcast.py


In [ ]:
!python -u /content/05_composition_broadcast.py

A descriptor: tensor<ptr<f32, gmem, align<16>> o (4):(1)>
transform layout: (4,3):(1,0)
B descriptor: tensor<ptr<f32, gmem, align<16>> o (4,3):(1,0)>
Original tensor A:
tensor(raw_ptr(0x0000786db7200000: f32, gmem, align<16>) o (4):(1), data=
       [ 0.000000, ],
       [ 1.000000, ],
       [ 2.000000, ],
       [ 3.000000, ])
Broadcast tensor B:
tensor(raw_ptr(0x0000786db7200000: f32, gmem, align<16>) o (4,3):(1,0), data=
       [[ 0.000000,  0.000000,  0.000000, ],
        [ 1.000000,  1.000000,  1.000000, ],
        [ 2.000000,  2.000000,  2.000000, ],
        [ 3.000000,  3.000000,  3.000000, ]])
B[2,0]=2.000000, B[2,1]=2.000000, B[2,2]=2.000000, A[2]=2.000000
PyTorch A remains unchanged: tensor([0., 1., 2., 3.], device='cuda:0')


In [ ]:
%%writefile /content/copy_kernel.py

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch


@cute.kernel
def copy_kernel(A: cute.Tensor, copy_atom: cute.TiledCopy):
    tidx, _, _ = cute.arch.thread_idx()

    smem = cutlass.utils.SmemAllocator()
    sA = smem.allocate_tensor(cutlass.Float32, cute.make_layout((20,)), 16)
    # sA.fill(1)

    # cute.printf(sA)

    thr_copy = copy_atom.get_slice(tidx)
    # print(thr_copy)
    tAgA = thr_copy.partition_S(A)
    tAsA = thr_copy.partition_D(sA)

    cute.printf(tAgA, tidx)
    # cute.printf(tAsA)

    cute.copy(copy_atom, tAgA, tAsA)

    cute.arch.cp_async_commit_group()
    cute.arch.cp_async_wait_group(0)

    # cute.printf(sA)

@cute.jit
def explain_copy(A: cute.Tensor):
    async_copy_atom_a = cute.make_copy_atom(
        cute.nvgpu.cpasync.CopyG2SOp(),
        cutlass.Float32,
        num_bits_per_copy=128 # 32
    )
    thr_layout = cute.make_layout((2,))   # change this thread and see the results vary
    val_layout = cute.make_layout((4,))   # 4 vals per thread
    tiled_copy = cute.make_tiled_copy_tv(async_copy_atom_a, thr_layout, val_layout)

    copy_kernel(A, tiled_copy).launch(
        grid=(1, 1, 1),
        block=(2, 1, 1),
    )

A = torch.arange(0, 20, device="cuda", dtype=torch.float32)
tensorA = from_dlpack(A, assumed_align=16)
explain_copy(tensorA)

Overwriting /content/copy_kernel.py


In [ ]:
!python  /content/copy_kernel.py

raw_ptr(0x000010f85d200000: f32, gmem, align<16>) o ((4,1),(3)):((1,0),(8)) = 
  ( 0.000000, 1.000000, 2.000000, 3.000000, 8.000000, 9.000000, 10.000000, 11.000000, 16.000000, 17.000000, 18.000000, 19.000000 ), 0
raw_ptr(0x000010f85d200010: f32, gmem, align<16>) o ((4,1),(3)):((1,0),(8)) = 
  ( 4.000000, 5.000000, 6.000000, 7.000000, 12.000000, 13.000000, 14.000000, 15.000000, 0.000000, 0.000000, 0.000000, 0.000000 ), 1


## Transpose of matrix using shared padded memory


In [1]:
!nvidia-smi
%pip install -q "nvidia-cutlass-dsl==4.7.0"

Wed Aug 26 19:30:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
import cutlass
import cutlass.cute as cute


@cute.kernel
def transpose_smem_padded_cute_kernel(
    tiled_input: cute.Tensor,
    tiled_output: cute.Tensor,
    tiled_input_coord: cute.Tensor,
    tiled_output_coord: cute.Tensor,
    input_shape: cute.Shape,
    output_shape: cute.Shape,
    tiled_copy: cute.TiledCopy,
    smem_layout: cute.Layout,
    smem_layout_t: cute.Layout,
):
    """CuTe-native 32x32, shared-memory-tiled FP32 transpose.

    The kernel does not manually derive a per-element (row, col) from
    threadIdx.  A TiledCopy partitions each CTA tile into the values owned by
    this thread.  The transpose itself is represented by a second layout over
    the same shared-memory allocation.
    """

    tidx, _, _ = cute.arch.thread_idx()
    block_x, block_y, _ = cute.arch.block_idx()

    # zipped_divide produced tensors with profile:
    #
    #   ((TileM, TileN), (RestM, RestN))
    #
    # Input block  (block_y, block_x) becomes output block
    # (block_x, block_y).
    input_tile_coord = ((None, None), (block_y, block_x))
    output_tile_coord = ((None, None), (block_x, block_y))

    gA = tiled_input[input_tile_coord]
    gB = tiled_output[output_tile_coord]
    cA = tiled_input_coord[input_tile_coord]
    cB = tiled_output_coord[output_tile_coord]

    # One allocation, two logical tensor views.
    #
    #   sA (r, c)  -> r * 33 + c
    #   sAT(r, c)  -> r      + c * 33 == sA(c, r)
    #
    # No data-moving transpose is performed inside SMEM.  Changing the layout
    # changes the coordinate-to-address map.
    allocator = cutlass.utils.SmemAllocator()
    sA = allocator.allocate_tensor(
        element_type=tiled_input.element_type,
        layout=smem_layout,
        byte_alignment=16,
        swizzle=None,
    )
    sAT = cute.make_tensor(sA.iterator, smem_layout_t)

    # The TiledCopy owns the mapping
    #
    #   (thread_id, value_id) -> (tile_row, tile_col).
    #
    # get_slice() selects this thread; partition_S/D applies that ownership to
    # tensors with completely different physical layouts.
    thr_copy = tiled_copy.get_slice(tidx)

    # ------------------------------------------------------------------
    # Phase 1: row-major GMEM tile -> padded row-major SMEM view.
    # ------------------------------------------------------------------
    tAgA = thr_copy.partition_S(gA)
    tAsA = thr_copy.partition_D(sA)

    # Canonical CuTe predication: partition an identity/coordinate tensor with
    # exactly the same copy partitioner used for the data tensor.
    tAcA = thr_copy.partition_S(cA)
    pred_A = cute.make_rmem_tensor(tAcA.shape, cutlass.Boolean)
    for i in range(cute.size(pred_A)):
        pred_A[i] = cute.elem_less(tAcA[i], input_shape)

    cute.copy(tiled_copy, tAgA, tAsA, pred=pred_A)

    # All threads must arrive, including threads whose complete tile fragment
    # was predicated off at a matrix edge.
    cute.arch.sync_threads()

    # ------------------------------------------------------------------
    # Phase 2: transposed SMEM view -> row-major output GMEM tile.
    # ------------------------------------------------------------------
    tBsAT = thr_copy.partition_S(sAT)
    tBgB = thr_copy.partition_D(gB)

    tBcB = thr_copy.partition_D(cB)
    pred_B = cute.make_rmem_tensor(tBcB.shape, cutlass.Boolean)
    for i in range(cute.size(pred_B)):
        pred_B[i] = cute.elem_less(tBcB[i], output_shape)

    cute.copy(tiled_copy, tBsAT, tBgB, pred=pred_B)


# input and output are row-major FP32 tensors on the GPU.
@cute.jit
def solve(
    input: cute.Tensor,
    output: cute.Tensor,
    rows: cute.Int32,
    cols: cute.Int32,
):
    tile_dim = 32
    block_rows = 8
    values_per_thread = tile_dim // block_rows
    padded_cols = tile_dim + 1

    # ------------------------------------------------------------------
    # 1. Describe ownership, instead of hand-writing index arithmetic.
    # ------------------------------------------------------------------
    # Thread layout:
    #   logical (thread_row, thread_col) -> thread_id
    #   (8, 32):(32, 1)
    #
    # Therefore every hardware warp occupies one logical thread row and its
    # 32 lanes span the tile's contiguous column mode.
    thr_layout = cute.make_ordered_layout(
        (block_rows, tile_dim),
        order=(1, 0),
    )

    # Each thread owns four values in the tile-row mode and one in tile-col.
    # Combining (8,32) threads with (4,1) values produces a (32,32) CTA tile.
    val_layout = cute.make_ordered_layout(
        (values_per_thread, 1),
        order=(1, 0),
    )

    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    # A scalar FP32 CopyUniversal atom is intentionally used in this learning
    # stage.  TiledCopy adds the thread/value ownership around the atom.
    copy_atom = cute.make_copy_atom(
        cute.nvgpu.CopyUniversalOp(),
        input.element_type,
        num_bits_per_copy=32,
    )
    tiled_copy = cute.make_tiled_copy_tv(
        copy_atom,
        thr_layout,
        val_layout,
    )

    # ------------------------------------------------------------------
    # 2. Tile the full tensors with layout algebra.
    # ------------------------------------------------------------------
    # zipped_divide groups the result as ((TileM,TileN),(RestM,RestN)).
    tiled_input = cute.zipped_divide(input, tiler_mn)
    tiled_output = cute.zipped_divide(output, tiler_mn)

    input_shape = (rows, cols)
    output_shape = (cols, rows)

    # Identity tensors carry logical coordinates, not matrix data.  Applying
    # the same divide and thread partition to them generates correct edge
    # predicates without reconstructing global indices by hand.
    input_coord = cute.make_identity_tensor(input_shape)
    output_coord = cute.make_identity_tensor(output_shape)
    tiled_input_coord = cute.zipped_divide(input_coord, tiler_mn)
    tiled_output_coord = cute.zipped_divide(output_coord, tiler_mn)

    # ------------------------------------------------------------------
    # 3. Give one SMEM allocation two coordinate-to-address maps.
    # ------------------------------------------------------------------
    # Both layouts have logical shape (32,32).  The physical row pitch is 33.
    smem_layout = cute.make_layout(
        (tile_dim, tile_dim),
        stride=(padded_cols, 1),
    )
    smem_layout_t = cute.make_layout(
        (tile_dim, tile_dim),
        stride=(1, padded_cols),
    )
    smem_bytes = cute.size_in_bytes(input.element_type, smem_layout)

    transpose_smem_padded_cute_kernel(
        tiled_input,
        tiled_output,
        tiled_input_coord,
        tiled_output_coord,
        input_shape,
        output_shape,
        tiled_copy,
        smem_layout,
        smem_layout_t,
    ).launch(
        grid=(
            cute.ceil_div(cols, tile_dim),
            cute.ceil_div(rows, tile_dim),
            1,
        ),
        # The thread count comes from the T mode of the TV layout: 8 * 32.
        block=(cute.size(tv_layout, mode=[0]), 1, 1),
        smem=smem_bytes,
    )


In [10]:
import torch
from cutlass.cute.runtime import from_dlpack

n = int(input("Enter n: "))

if n <= 0:
    raise ValueError("n must be positive")

# FP32 row-major GPU matrix:
#
# [[1,       2, ..., n],
#  [n+1,   n+2, ..., 2n],
#  ...
#  [...,          n*n]]
input_gpu = torch.arange(
    1,
    n * n + 1,
    dtype=torch.float32,
    device="cuda",
).reshape(n, n).contiguous()

output_gpu = torch.empty_like(input_gpu)

# Convert PyTorch GPU tensors to CuTe tensors.
input_cute = from_dlpack(input_gpu, assumed_align=16)
output_cute = from_dlpack(output_gpu, assumed_align=16)

rows = cutlass.Int32(n)
cols = cutlass.Int32(n)

print("Compiling CuTe kernel...")

compiled_solve = cute.compile(
    solve,
    input_cute,
    output_cute,
    rows,
    cols,
)

print("Running CuTe kernel...")

compiled_solve(
    input_cute,
    output_cute,
    rows,
    cols,
)

torch.cuda.synchronize()

# Convert only for cleaner integer-style display. The GPU tensors remain FP32.
print("\nInput matrix:")
print(input_gpu.cpu().to(torch.int64))

print("\nTransposed output:")
print(output_gpu.cpu().to(torch.int64))

expected = input_gpu.T.contiguous()
torch.testing.assert_close(output_gpu, expected, rtol=0, atol=0)

print("\nResult is correct:", torch.equal(output_gpu, expected))

Enter n: 50
Compiling CuTe kernel...
Running CuTe kernel...

Input matrix:
tensor([[   1,    2,    3,  ...,   48,   49,   50],
        [  51,   52,   53,  ...,   98,   99,  100],
        [ 101,  102,  103,  ...,  148,  149,  150],
        ...,
        [2351, 2352, 2353,  ..., 2398, 2399, 2400],
        [2401, 2402, 2403,  ..., 2448, 2449, 2450],
        [2451, 2452, 2453,  ..., 2498, 2499, 2500]])

Transposed output:
tensor([[   1,   51,  101,  ..., 2351, 2401, 2451],
        [   2,   52,  102,  ..., 2352, 2402, 2452],
        [   3,   53,  103,  ..., 2353, 2403, 2453],
        ...,
        [  48,   98,  148,  ..., 2398, 2448, 2498],
        [  49,   99,  149,  ..., 2399, 2449, 2499],
        [  50,  100,  150,  ..., 2400, 2450, 2500]])

Result is correct: True


# CUDA Shared-Memory Bank Conflicts — Learning Notes

* NVIDIA shared memory has **32 banks**.

* Bank conflicts are checked **per warp, per shared-memory instruction**.

* A block with 64 threads has **2 warps**; conflicts are calculated separately for each warp.

* For FP32:

  `bank = element_address % 32`

* A conflict happens when threads access **different addresses in the same bank**.

* Reading the exact same address is a broadcast, not a conflict.

## 32×32 tile: column access

| Physical row pitch | Result          |
| -----------------: | --------------- |
|                 32 | 32-way conflict |
|          33 (`+1`) | No conflict     |
|          34 (`+2`) | 2-way conflict  |
|          35 (`+3`) | No conflict     |
|          36 (`+4`) | 4-way conflict  |

For 32 active lanes:

`conflict degree = gcd(row pitch, 32)`

Padding by `+1` is commonly used because it removes the conflict while using the least extra shared memory.

## 16×16 tile

If 16 threads read one column with a row pitch of 16:

`banks = 0, 16, 0, 16, ...`

Eight threads access each bank, giving an **8-way conflict**.

Padding the row pitch to 17 removes the conflict for those 16 threads. However, if a full warp accesses two columns at once, the exact result depends on the thread-to-element mapping. For a common `(16, 4)` thread arrangement:

| Physical row pitch | Result         |
| -----------------: | -------------- |
|                 16 | 8-way conflict |
|                 17 | 2-way conflict |
|                 18 | No conflict    |

## Main takeaway

A tile size alone does not determine the conflict. Always examine the shared-memory addresses accessed by all 32 lanes during one instruction.
